In [113]:
import os
import warnings
from pathlib import Path

import librosa
import torch
from IPython.display import Audio
from dotenv import load_dotenv
from pyannote.audio import Pipeline

load_dotenv()

True

In [124]:
SAMPLE_RATE = 16000

In [12]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=os.environ["HUGGINGFACE_TOKEN"])

In [94]:
def load_audio_as_mapping(audio_path: str|Path) -> dict:
    waveform, samplerate = librosa.load(audio_path, sr=SAMPLE_RATE)

    waveform = torch.from_numpy(waveform).unsqueeze(0).float()

    # Compatible with pyannote, and allows us to easily split waveform into smaller chunks
    audio_mapping = {
        "waveform": waveform,
        "sample_rate": samplerate,
        "channel": 0,
        "uri": audio_path.name,
    }
    return audio_mapping

In [115]:
Audio(audio["waveform"], rate=16000)

In [121]:
with warnings.catch_warnings(action="ignore"):
    diary = pipeline(audio)

In [122]:
annotation = diary.speaker_diarization

for turn, _, speaker in annotation.itertracks(yield_label=True):
    print(f"[{turn.start:.2f}s - {turn.end:.2f}s] {speaker}")

[0.03s - 1.38s] SPEAKER_00
[1.79s - 3.51s] SPEAKER_00


In [138]:
# Slicing the audio to individual speaker segments
segments = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    start_idx = int(turn.start * SAMPLE_RATE)
    end_idx = int(turn.end * SAMPLE_RATE)
    segment = audio["waveform"][0, start_idx:end_idx]
    segments.append(segment)

In [140]:
Audio(segments[0], rate=SAMPLE_RATE)

In [141]:
Audio(segments[1], rate=SAMPLE_RATE)